# 09 — Endpoint vLLM temporaire pour la soutenance

**Statut : procédure préparée, vérification cloud publique à exécuter.**

[← 08 · Résultat du DPO ciblé](08_dpo_triage_cible_colab.ipynb) · [Sommaire](../README.md)

Je complète la preuve d’inférence interne du notebook 06 par un appel à une URL HTTPS publique, protégée par clé. Ce service utilise réellement **FastAPI + vLLM**, avec le SFT successif v3 déjà évalué ; le Space Gradio reste une autre voie, sous Transformers/ZeroGPU.

```text
Navigateur → HTTPS temporaire (Cloudflare) → FastAPI / Bearer
                                            ↓ UUID commun
                                         vLLM → contrôle → SQLite → réponse
```

Je choisis une A100 ou G4, autorise Drive et passe `LANCER_ENDPOINT` à `True`. La cellule installe l’environnement, vérifie les poids et le binaire du tunnel, puis demande une clé masquée (au moins 24 caractères). Je garde cette clé pour la saisir dans la page de démo ; je ne la partage pas dans les logs, le chat ou le support public.

Le paquet `ocr14_endpoint_soutenance_20261106.zip` doit être dans mon dossier Drive `OCR_PROJET_14`. Le lancement affiche `DEMO_URL=...` après trois appels fictifs vérifiés à travers l’URL HTTPS. J’ouvre cette URL, saisis la clé et montre un parcours. Les journaux enregistrent les appels avec leurs identifiants, sans clé ni corps dans les logs techniques.

**Durée et coût.** Le GPU reste alloué pendant installation, chargement et démonstration. La fenêtre de démo est limitée à 20 minutes après vérification. Interrompre la cellule déclenche l’arrêt des services, la sauvegarde et la libération du GPU. Ce n’est pas un hébergement permanent ; le tunnel ne garantit pas de disponibilité et son URL change à chaque lancement.

Je ne présente le livrable endpoint public comme vérifié qu’après lecture du rapport `verification_publique.json` et un essai depuis un navigateur extérieur. Si le tunnel ou vLLM échoue, je garde le statut d’échec et les logs. Les pages d’explication du Space restent accessibles sans cette session.

Sources : [Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) · [vLLM](https://docs.vllm.ai/en/latest/serving/openai_compatible_server/).


In [ ]:
LANCER_ENDPOINT = False  # True uniquement pour une démonstration GPU.
DUREE_DEMO_MINUTES = 20  # Arrêt automatique après vérification ; 1 à 60 minutes.
if LANCER_ENDPOINT:
    from google.colab import drive, runtime
    from getpass import getpass
    from pathlib import Path
    from datetime import datetime, timezone
    import hashlib, os, shutil, signal, subprocess, sys, urllib.request, zipfile
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/OCR_PROJET_14')
    RACINE = Path('/content/OCR14_endpoint_soutenance')
    RACINE.mkdir(parents=True, exist_ok=True)
    identifiant = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    SORTIES = Path('/content') / f'endpoint_soutenance_{identifiant}'
    processus = None
    try:
        print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip())
        paquet = DRIVE / 'ocr14_endpoint_soutenance_20261106.zip'
        assert hashlib.sha256(paquet.read_bytes()).hexdigest() == 'a78169fc13e38942b58874704b7fb82579a49fe12d3d7201150409f7f016f0e2', 'Paquet différent'
        with zipfile.ZipFile(paquet) as z:
            assert all((RACINE / n).resolve().is_relative_to(RACINE.resolve()) for n in z.namelist())
            z.extractall(RACINE)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        ENV = '/content/ocr14_endpoint_env'
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', '--managed-python', ENV], check=True)
        PYTHON = ENV + '/bin/python'
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
                        'vllm==0.32.0', 'uvicorn==0.54.0', 'jinja2==3.1.6', '--torch-backend=auto'], check=True)
        subprocess.run([ENV + '/bin/vllm', 'download-kernels'], check=True)
        # Poids publics déjà déployés, révision figée ; le lanceur vérifie aussi leur SHA.
        telecharger = "from huggingface_hub import hf_hub_download; " + "; ".join(
            f"hf_hub_download('rayakevin/ocr14-triage-fr-en', 'adapter/{nom}', repo_type='space', revision='b467327ca9c63369a44021827e052ba28ae47861', local_dir='{RACINE / 'poids'}')"
            for nom in ('adapter_config.json', 'adapter_model.safetensors'))
        subprocess.run([PYTHON, '-c', telecharger], check=True)
        tunnel = RACINE / 'cloudflared'
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64', tunnel)
        assert hashlib.sha256(tunnel.read_bytes()).hexdigest() == '1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'
        tunnel.chmod(0o700)
        cle = getpass('Clé temporaire (24 caractères minimum ; à saisir ensuite dans la page de démo) : ')
        if len(cle) < 24:
            raise ValueError('Clé trop courte ; aucune API exposée.')
        commande = [PYTHON, '-m', 'serving.servir_demo', '--adapter', str(RACINE / 'poids/adapter'),
                    '--cloudflared', str(tunnel), '--output', str(SORTIES), '--minutes', str(DUREE_DEMO_MINUTES)]
        processus = subprocess.Popen(commande, cwd=RACINE, env={**os.environ, 'OCR14_API_KEY': cle, 'PYTHONUNBUFFERED': '1'},
                                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
        del cle
        for ligne in processus.stdout:
            print(ligne, end='', flush=True)
        if processus.wait():
            raise RuntimeError('Échec de la démonstration ; voir les journaux sauvegardés.')
    finally:
        if processus is not None and processus.poll() is None:
            # SIGINT laisse le lanceur fermer vLLM, l’API et le tunnel dans son finally.
            os.killpg(processus.pid, signal.SIGINT)
            try: processus.wait(timeout=90)
            except subprocess.TimeoutExpired:
                os.killpg(processus.pid, signal.SIGKILL)
                processus.wait()
        if SORTIES.exists():
            destination = DRIVE / 'evaluations' / SORTIES.name
            shutil.copytree(SORTIES, destination, dirs_exist_ok=True)
            for f in SORTIES.rglob('*'):
                if f.is_file():
                    assert hashlib.sha256(f.read_bytes()).digest() == hashlib.sha256((destination / f.relative_to(SORTIES)).read_bytes()).digest(), 'Sauvegarde incomplète : je conserve la VM'
            print('Preuves sauvegardées dans', destination)
        print('Synchronisation Drive et libération du GPU.')
        drive.flush_and_unmount(timeout_ms=300_000)
        runtime.unassign()
else:
    print('Endpoint désactivé. Aucun GPU ni tunnel démarré.')
